# 8. Submodules

A **submodule** is a repository inside another repository. The outer
project does not store the inner project's files. It stores only **which
commit** of it to use.

```mermaid
flowchart LR
    subgraph barge["barge (your project)"]
        src["hydrostatics.py"]
        pin["vendor/seawater (one commit id)"]
    end
    pin -.->|points at| lib["seawater repository"]
```

Our example: the sea water properties live in their own repository,
`seawater`, maintained by another group. The barge project uses it.

In [1]:
!rm -rf ~/git-course/08-submodules
!mkdir -p ~/git-course/08-submodules
%cd ~/git-course/08-submodules

/home/emircem/git-course/08-submodules


## The library

The hydrography group's repository, with its server `seawater.git`:

In [2]:
!git init --bare seawater.git
!mkdir seawater
%cd seawater

Initialized empty Git repository in /home/emircem/git-course/08-submodules/seawater.git/
/home/emircem/git-course/08-submodules/seawater


In [3]:
%%writefile seawater.py
"""Properties of sea water."""


def density(temperature=15.0):
    """Density of sea water in kg/m^3."""
    return 1028.0 - 0.2 * temperature

Writing seawater.py


In [4]:
!git init
!git add seawater.py
!git commit -m "Add sea water density"
!git remote add origin ../seawater.git
!git push -u origin main

Initialized empty Git repository in /home/emircem/git-course/08-submodules/seawater/.git/
[main (root-commit) f7f7771] Add sea water density
 1 file changed, 6 insertions(+)
 create mode 100644 seawater.py
Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 335 bytes | 335.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To ../seawater.git
 * [new branch]      main -> main
branch 'main' set up to track 'origin/main'.


## Our project

In [5]:
%cd ..
!git init --bare barge.git
!mkdir barge
%cd barge

/home/emircem/git-course/08-submodules
Initialized empty Git repository in /home/emircem/git-course/08-submodules/barge.git/
/home/emircem/git-course/08-submodules/barge


In [6]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft

Writing hydrostatics.py


In [7]:
!git init
!git add hydrostatics.py
!git commit -m "Add volume"
!git remote add origin ../barge.git
!git push -u origin main

Initialized empty Git repository in /home/emircem/git-course/08-submodules/barge/.git/
[main (root-commit) c60a72a] Add volume
 1 file changed, 6 insertions(+)
 create mode 100644 hydrostatics.py
Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 340 bytes | 340.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To ../barge.git
 * [new branch]      main -> main
branch 'main' set up to track 'origin/main'.


## Adding the submodule

`git submodule add <address> <folder>` puts the library in a folder:

In [8]:
!git -c protocol.file.allow=always submodule add ../seawater.git vendor/seawater

Cloning into '/home/emircem/git-course/08-submodules/barge/vendor/seawater'...
done.


About `-c protocol.file.allow=always`: our "server" is a local folder, and
for safety git does not allow submodules from local folders unless you say
so. With a real address (`https://github.com/...`) you don't need it.

In [9]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   .gitmodules
	new file:   vendor/seawater



Two new things: the folder, and a file `.gitmodules` that remembers where
the library comes from:

In [10]:
!cat .gitmodules

[submodule "vendor/seawater"]
	path = vendor/seawater
	url = ../seawater.git


In [11]:
!git commit -m "Add the seawater library as a submodule"
!git push

[main 2c1b747] Add the seawater library as a submodule
 2 files changed, 4 insertions(+)
 create mode 100644 .gitmodules
 create mode 160000 vendor/seawater
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 20 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 442 bytes | 442.00 KiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To ../barge.git
   c60a72a..2c1b747  main -> main


## What was actually saved

`git ls-tree -r` lists everything a commit contains:

In [12]:
!git ls-tree -r HEAD

100644 blob 8abc87aaf05274e55fae7f21aec3af8d22e6c8bc	.gitmodules
100644 blob 6d216b8fcbbc8b4bbc7f55df98effd6795d0d795	hydrostatics.py
160000 commit f7f7771e86e86630b99b0939f92c20967312b9e4	vendor/seawater


`hydrostatics.py` is a `blob` (a file). `vendor/seawater` is a `commit`:
the project saved **one commit id**, not the library's files. That id pins
the exact version of the library we use.

In [13]:
!git submodule status

 f7f7771e86e86630b99b0939f92c20967312b9e4 vendor/seawater (heads/main)


## Surprise: a fresh clone is empty

A colleague clones our project:

In [14]:
%cd ..
!git clone barge.git colleague

/home/emircem/git-course/08-submodules
Cloning into 'colleague'...
done.


In [15]:
!ls colleague/vendor/seawater

The folder is **empty**. Cloning the project does not download its
submodules. You have to ask:

In [16]:
%cd colleague
!git -c protocol.file.allow=always submodule update --init

/home/emircem/git-course/08-submodules/colleague
Submodule 'vendor/seawater' (/home/emircem/git-course/08-submodules/seawater.git) registered for path 'vendor/seawater'
Cloning into '/home/emircem/git-course/08-submodules/colleague/vendor/seawater'...
done.
Submodule path 'vendor/seawater': checked out 'f7f7771e86e86630b99b0939f92c20967312b9e4'


In [17]:
!ls vendor/seawater

seawater.py


Tip: `git clone --recurse-submodules <address>` does both steps at once.

## The library gets an update

The hydrography group improves their formula and pushes:

In [18]:
%cd ../seawater

/home/emircem/git-course/08-submodules/seawater


In [19]:
%%writefile seawater.py
"""Properties of sea water."""


def density(temperature=15.0, salinity=35.0):
    """Density of sea water in kg/m^3."""
    return 1000.0 + 0.8 * salinity - 0.2 * temperature

Overwriting seawater.py


In [20]:
!git add seawater.py
!git commit -m "Take salinity into account"
!git push

[main 7534ba2] Take salinity into account
 1 file changed, 2 insertions(+), 2 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 335 bytes | 335.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To ../seawater.git
   f7f7771..7534ba2  main -> main


Our project is not affected: it still uses the old commit. That is the
point of pinning — nothing changes under you without your say.

In [21]:
%cd ../barge
!git submodule status

/home/emircem/git-course/08-submodules/barge
 f7f7771e86e86630b99b0939f92c20967312b9e4 vendor/seawater (heads/main)


To take the new version, update the submodule:

In [22]:
!git -c protocol.file.allow=always submodule update --remote

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 315 bytes | 315.00 KiB/s, done.
From /home/emircem/git-course/08-submodules/barge/../seawater
   f7f7771..7534ba2  main       -> origin/main
Submodule path 'vendor/seawater': checked out '7534ba2f9944fad428709f0a6f0a6023cd4856d1'


In [23]:
!git diff

diff --git a/vendor/seawater b/vendor/seawater
index f7f7771..7534ba2 160000
--- a/vendor/seawater
+++ b/vendor/seawater
@@ -1 +1 @@
-Subproject commit f7f7771e86e86630b99b0939f92c20967312b9e4
+Subproject commit 7534ba2f9944fad428709f0a6f0a6023cd4856d1


The only change in our project: the commit id. Save it like any change:

In [24]:
!git add vendor/seawater
!git commit -m "Update the seawater library"
!git push

[main bde1c23] Update the seawater library
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 300 bytes | 300.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To ../barge.git
   2c1b747..bde1c23  main -> main


## Summary

| I want to… | Command |
| - | - |
| add a submodule | `git submodule add <address> <folder>` |
| clone a project with its submodules | `git clone --recurse-submodules <address>` |
| fill in submodules after a plain clone | `git submodule update --init` |
| see which commits are pinned | `git submodule status` |
| move to the library's newest commit | `git submodule update --remote` |

Submodules add steps for everyone. Use them when you need to know exactly
which version of a library produced your results. For published Python
packages (`numpy`, `scipy`), use `pip` and a `requirements.txt` instead.

## Exercises

1. In `colleague`, run `git pull` and then
   `git -c protocol.file.allow=always submodule update`. Does it get the
   new library version?
2. Run `cat vendor/seawater/seawater.py` in `barge` before and after the
   update.

That's the course! The [cheat sheet](../docs/09-cheat-sheet.md) has every
command on one page.